In [1]:
# %pip install lightautoml

In [1]:
import pandas as pd
import numpy as np
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task
from sklearn.metrics import mean_absolute_percentage_error

/var/folders/q_/fz2ylcb14dqcs8bp2t8hjz6m0000gn/T/ipykernel_23897/1255200142.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


'nlp' extra dependency package 'gensim' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'gensim' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/lightautoml/ml_algo/dl_model.py:42: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/lightautoml/text/embed.py:22: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/lightautoml/text/dl_transformers.py:25: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")


In [2]:
import pandas as pd
import numpy as np
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task
from sklearn.metrics import mean_absolute_percentage_error

class StressTestLAMA:
    def __init__(self, fund_file: str, moex_file: str, rgbitr_file: str,
                 moex_stress_file: str, rgbitr_stress_file: str):
        self.fund_file = fund_file
        self.moex_file = moex_file
        self.rgbitr_file = rgbitr_file
        self.moex_stress_file = moex_stress_file
        self.rgbitr_stress_file = rgbitr_stress_file
        self.model = None
        self.mape_score = None
        self.train_data = None
        self.stress_result = None

    @staticmethod
    def preprocess_price_file(filepath: str, column_name: str) -> pd.Series:
        df = pd.read_excel(filepath)
        df.columns = df.columns.astype(str)
        df[df.columns[0]] = pd.to_datetime(df[df.columns[0]])
        df.set_index(df.columns[0], inplace=True)
        df.index = df.index.normalize()

        raw_series = df.iloc[:, 0].astype(str).str.replace(" ", "").str.replace(",", ".")
        series = pd.to_numeric(raw_series, errors="coerce")
        series.name = column_name
        return series

    def _load_and_prepare(self):
        fund = self.preprocess_price_file(self.fund_file, "Fund")
        moex = self.preprocess_price_file(self.moex_file, "IMOEX")
        rgbitr = self.preprocess_price_file(self.rgbitr_file, "RGBITR")
        df = pd.concat([fund, moex, rgbitr], axis=1).dropna()
        self.train_data = df.pct_change().dropna()

    def _load_stress_data(self):
        moex = self.preprocess_price_file(self.moex_stress_file, "IMOEX")
        rgbitr = self.preprocess_price_file(self.rgbitr_stress_file, "RGBITR")
        df = pd.concat([moex, rgbitr], axis=1).dropna()
        return df.pct_change().dropna()

    def train_model(self, timeout=300):
        self._load_and_prepare()
        task = Task("reg")
        roles = {"target": "Fund"}
        automl = TabularAutoML(task=task, timeout=timeout, cpu_limit=2, reader_params={"n_jobs": 1})
        oof_pred = automl.fit_predict(self.train_data, roles=roles)
        self.model = automl
        self.mape_score = mean_absolute_percentage_error(self.train_data["Fund"], oof_pred.data)
        return self.mape_score

    def predict_stress(self):
        stress_data = self._load_stress_data()
        pred = self.model.predict(stress_data)
        self.stress_result = pd.Series(pred.data.ravel(), index=stress_data.index)
        return self.stress_result

    def get_summary(self):
        cum_return = (1 + self.stress_result).prod() - 1
        rolling_max = (1 + self.stress_result).cumprod().cummax()
        drawdown = (1 + self.stress_result).cumprod() / rolling_max - 1
        max_drawdown = drawdown.min()
        return {
            "MAPE (train)": self.mape_score,
            "Cumulative return (stress)": cum_return,
            "Max drawdown (stress)": max_drawdown
        }

In [3]:

# Теперь повторно запустим стресс-тест
fund_path = "data/test_new_opportunities.xlsx"
moex_path = "data/moex_train.xlsx"
rgbitr_path = "data/rgbitr_train.xlsx"
moex_stress_path = "data/moex_stress.xlsx"
rgbitr_stress_path = "data/rgbitr_stress.xlsx"



In [5]:
model = StressTestLAMA(fund_path, moex_path, rgbitr_path, moex_stress_path, rgbitr_stress_path)
mape = model.train_model()
stress_predictions = model.predict_stress()
summary = model.get_summary()



In [6]:
summary

{'MAPE (train)': 1.8790386736433522,
 'Cumulative return (stress)': -0.046787261962890625,
 'Max drawdown (stress)': -0.13812059}

In [4]:
fund_path = "/Users/contessina/Documents/my_projects/RR/data/test_local.xlsx"
model = StressTestLAMA(fund_path, moex_path, rgbitr_path, moex_stress_path, rgbitr_stress_path)
mape = model.train_model()
stress_predictions = model.predict_stress()
summary = model.get_summary()
summary

{'MAPE (train)': 4321693653.75168,
 'Cumulative return (stress)': 0.16566252708435059,
 'Max drawdown (stress)': -0.054973304}

In [5]:
fund_path = "/Users/contessina/Documents/my_projects/RR/data/test_russian_bonds.xlsx"
model = StressTestLAMA(fund_path, moex_path, rgbitr_path, moex_stress_path, rgbitr_stress_path)
mape = model.train_model()
stress_predictions = model.predict_stress()
summary = model.get_summary()
summary

{'MAPE (train)': 9.876901648409484,
 'Cumulative return (stress)': 0.10961282253265381,
 'Max drawdown (stress)': -0.014359176}

In [6]:
fund_path = "/Users/contessina/Documents/my_projects/RR/data/test_russian_shares.xlsx"
model = StressTestLAMA(fund_path, moex_path, rgbitr_path, moex_stress_path, rgbitr_stress_path)
mape = model.train_model()
stress_predictions = model.predict_stress()
summary = model.get_summary()
summary

{'MAPE (train)': 1.6125771443884174,
 'Cumulative return (stress)': 0.22524213790893555,
 'Max drawdown (stress)': -0.15534663}

In [ ]:
fund_path = "/Users/contessina/Documents/my_projects/RR/data/test_local.xlsx"
model = StressTestLAMA(fund_path, moex_path, rgbitr_path, moex_stress_path, rgbitr_stress_path)
mape = model.train_model()
stress_predictions = model.predict_stress()
summary = model.get_summary()
summary